# Vídeo «lectura» (sin GPU)

El transcript en pantalla, sincronizado frase a frase con el master: la frase que suena en grande, quién habla, la
anterior y la siguiente atenuadas, y una barra de progreso. Los tiempos salen de `indice_master.tsv` (lo escribió la
fábrica en la carpeta de Drive de cada episodio), así que no hace falta Whisper ni GPU.

**Elige un entorno sin acelerador** (*Entorno de ejecución → Cambiar tipo → CPU*) y ejecuta todo seguido. El vídeo
queda en `<carpeta del episodio>/video/` junto con su `.srt`. Plan: `docs/video-dinamico.md`.

In [ ]:
# 1. Preparación: fuente Inter, repo y Drive
import os, sys, subprocess
from pathlib import Path
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA,
                    "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)

def hay_inter():
    r = subprocess.run(["fc-match", "Inter:style=SemiBold", "-f", "%{file}"], capture_output=True, text=True)
    return "inter" in Path(r.stdout).name.lower()

if not hay_inter():
    subprocess.run("apt-get -qq install -y fonts-inter > /dev/null 2>&1", shell=True)
if not hay_inter():   # si el paquete no existe en esta imagen de Colab, la fuente oficial (licencia OFL)
    subprocess.run("cd /tmp && wget -q https://github.com/rsms/inter/releases/download/v4.1/Inter-4.1.zip "
                   "&& unzip -q -o Inter-4.1.zip -d inter && mkdir -p ~/.fonts "
                   "&& cp inter/extras/otf/Inter-*.otf ~/.fonts/ && fc-cache -f", shell=True, check=True)
assert hay_inter(), "no se ha podido instalar la fuente Inter"

sys.path[:0] = ["/content/repo/fabrica", "/content/repo/piloto", "/content/repo/criba"]
import comun, fabrica
comun.montar_drive()
CFG = fabrica.cargar_config()
RAIZ_EPISODIOS = Path(CFG["rutas"]["drive"]) / CFG["rutas"]["episodios"]
print("listo")

In [ ]:
# 2. Parámetros
EPISODIOS = ["2020/053-philosopher"]   # carpetas del repo (año/NNN-slug)
DURACION = None                        # None = entero; un número = solo los primeros N segundos (para probar)

In [ ]:
# 3. Renderizar
import json
from IPython.display import Video, display
hechos = []
for ep in EPISODIOS:
    titulo_en = json.loads(Path(f"/content/repo/episodios/{ep}/metadata.en.json").read_text(encoding="utf-8"))["titulo"]
    carpeta = fabrica.carpeta_episodio(RAIZ_EPISODIOS, ep, titulo_en, crear=False)
    print(f"— {ep}  ({carpeta})")
    cmd = [sys.executable, "/content/repo/scripts/video_lectura.py", ep, "--carpeta", str(carpeta)]
    if DURACION:
        cmd += ["--duracion", str(DURACION)]
    subprocess.run(cmd, check=True)
    hechos += sorted((carpeta / "video").glob("*.lectura.mp4"), key=lambda p: p.stat().st_mtime)[-1:]

# el último, para verlo aquí mismo (solo si es corto: Colab no embebe bien vídeos grandes)
if hechos and hechos[-1].stat().st_size < 60e6:
    display(Video(str(hechos[-1]), embed=True, width=960))